In [ ]:
# ---------- Importe ----------
import json # zum Speichern der Daten als JSON-Datei
import os # für Ordner und Dateipfade
import pandas as pd # für die Tabellen

In [ ]:
# Überprüfung Semaglutid Juli 2025
df = pd.read_csv('data/raw/berichte_pro_tag.csv', parse_dates=['time'])  # CSV-Datei einlesen und die Spalte "time" als Datum interpretieren
df.head()

In [ ]:
df.dtypes

In [ ]:
semaglutide_mask = df['wirkstoff'] == 'semaglutide'
mask_juni = (df['time'] >= '2025-06-01') & (df['time'] <= '2025-06-30')
df[semaglutide_mask & mask_juni].groupby('time')['count'].sum() # brauche eigentlich nicht zu gruppieren, aber mir gefällt die Ansicht besser

In [ ]:
semaglutide_mask = df['wirkstoff'] == 'semaglutide'
mask_juli = (df['time'] >= '2025-07-01') & (df['time'] <= '2025-07-31')
df[semaglutide_mask & mask_juli].groupby('time')['count'].sum()

In [ ]:
with open('data/raw/semaglutide_berichte_20250701_bis_20250731.json', 'r') as f:
    berichte = json.load(f)

In [ ]:
len(berichte)

In [ ]:
df_juli = pd.DataFrame()   # leere Tabelle anlegen

for i in range(len(berichte)):   # für jeden Bericht (wie range(df.shape[0]))
    bericht_dict = berichte[i]   # ein Bericht als Dictionary (wie response_dict)
    quelle = bericht_dict.get('primarysource') or {}   # Infos zur meldenden Person, leer falls nicht vorhanden

    df_juli.loc[i, 'id'] = bericht_dict['safetyreportid']   # eindeutige Nummer des Berichts
    df_juli.loc[i, 'datum'] = bericht_dict['receivedate']    # Tag an dem FDA Bericht erhalten hat
    df_juli.loc[i, 'meldende'] = quelle.get('qualification') # Wer den Fall urspünglich gemeldet hat (z.b. 1= Arzt/Ärztin)
    df_juli.loc[i, 'land'] = bericht_dict.get('occurcountry') # Land, in dem die Nebenwirkungen aufgetreten sind
    df_juli.loc[i, 'berichtsart'] = bericht_dict.get('reporttype') # Woher der Bericht stammt (z.B. 1 = Spontanmeldung)
    df_juli.loc[i, 'firmennummer'] = bericht_dict.get('companynumb') # Fallnummer des Herstellers, der den Fall weitergeleitet hat

df_juli.head()

In [ ]:
mask_ausreisser = df_juli['datum'].isin(['20250715', '20250716', '20250718'])
df_juli[mask_ausreisser]['meldende'].value_counts(normalize=True)

In [ ]:
df_juli[~mask_ausreisser]['meldende'].value_counts(normalize=True)

In [ ]:
mask_ausreisser = df_juli['datum'].isin(['20250715', '20250716', '20250718'])
df_juli[mask_ausreisser]['land'].value_counts(normalize=True)

In [ ]:
df_juli[~mask_ausreisser]['land'].value_counts(normalize=True)

In [ ]:
mask_ausreisser = df_juli['datum'].isin(['20250715', '20250716', '20250718'])
df_juli[mask_ausreisser]['berichtsart'].value_counts(normalize=True)

In [ ]:
df_juli[~mask_ausreisser]['berichtsart'].value_counts(normalize=True)

In [ ]:
df_juli['firma'] = df_juli['firmennummer'].str.split('-').str[1]   # Firmenkürzel aus der Firmennummer herausziehen
df_juli[mask_ausreisser]['firma'].value_counts(normalize=True).head(10)    # Firmen an den Ausreißer-Tagen (Anteile, Top 10)

In [ ]:
df_juli[~mask_ausreisser]['firma'].value_counts(normalize=True).head(10)   # Firmen an den übrigen Tagen (Anteile, Top 10)

In [ ]:
ordner = 'data/raw'   # hier liegen die Rohdaten

# ---------- Alle JSON-Dateien im Ordner finden ----------
dateien = []   # leere Liste für die Dateinamen
for name in os.listdir(ordner):   # alle Dateien im Ordner durchgehen
    if name.endswith('.json'):    # nur JSON-Dateien (die CSV der Zeitreihe nicht)
        dateien.append(name)      # Dateinamen merken
print(len(dateien), 'Dateien gefunden')   # sollten 24 sein

In [ ]:
# ---------- Leere Listen für die drei Tabellen ----------
berichte_zeilen = []         # eine Zeile pro Bericht
nebenwirkungen_zeilen = []   # eine Zeile pro gemeldeter Nebenwirkung
medikamente_zeilen = []      # eine Zeile pro genanntem Medikament

# ---------- Durch alle Dateien und Berichte gehen ----------
for name in dateien:   # jede Monatsdatei
    abfrage = name.split('_')[0]   # aus 'semaglutide_berichte_...' wird 'semaglutide'
    with open(os.path.join(ordner, name), 'r') as f:   # Datei öffnen (os.path.join setzt Ordner + Name zusammen)
        berichte = json.load(f)   # Liste mit allen Berichten dieses Monats

    for bericht in berichte:   # jeden Bericht durchgehen
        bericht_id = bericht.get('safetyreportid')   # ID merken, sie verbindet später die drei Tabellen
        patient = bericht.get('patient') or {}       # Teil mit Patientendaten (leer, falls er fehlt)
        quelle = bericht.get('primarysource') or {}  # Teil mit der meldenden Person (leer, falls er fehlt)

        # --- Tabelle 1: Berichte ---
        berichte_zeilen.append({
            'id': bericht_id,
            'abfrage': abfrage,                                  # aus welchem Download der Bericht stammt
            'datum': bericht.get('receivedate'),                 # Eingangsdatum
            'schwerwiegend': bericht.get('serious'),             # 1 = ja, 2 = nein
            'geschlecht': patient.get('patientsex'),             # 0 = unbekannt, 1 = männlich, 2 = weiblich
            'alter': patient.get('patientonsetage'),             # Alter (Zahl)
            'alter_einheit': patient.get('patientonsetageunit'), # Einheit des Alters (Code)
            'meldende': quelle.get('qualification'),             # wer gemeldet hat
            'land': bericht.get('occurcountry'),                 # Land
            'berichtsart': bericht.get('reporttype'),            # Art des Berichts
            'firmennummer': bericht.get('companynumb'),          # Fallnummer des Herstellers
        })

# --- Tabelle 2: Nebenwirkungen ---
        for reaktion in patient.get('reaction', []):   # jede Nebenwirkung im Bericht (leere Liste, falls keine)
            nebenwirkungen_zeilen.append({
                'id': bericht_id,
                'abfrage': abfrage,                                 # aus welchem Download die Zeile stammt
                'nebenwirkung': reaktion.get('reactionmeddrapt'),   # Name der Nebenwirkung
            })

        # --- Tabelle 3: Medikamente ---
        for medikament in patient.get('drug', []):   # jedes Medikament im Bericht
            wirkstoff_info = medikament.get('activesubstance') or {}   # verschachtelter Teil mit dem Wirkstoff
            medikamente_zeilen.append({
                'id': bericht_id,
                'abfrage': abfrage,                                         # aus welchem Download die Zeile stammt
                'medikament': medikament.get('medicinalproduct'),          # eingetragener Name
                'wirkstoff': wirkstoff_info.get('activesubstancename'),     # Wirkstoff
                'rolle': medikament.get('drugcharacterization'),            # 1 = verdächtig, 2 = Begleitmedikation, 3 = Wechselwirkung
                'anwendung': medikament.get('drugindication'),              # wofür es genommen wurde
            })

# ---------- Aus den Listen Tabellen machen ----------
df_berichte = pd.DataFrame(berichte_zeilen)
df_nebenwirkungen = pd.DataFrame(nebenwirkungen_zeilen)
df_medikamente = pd.DataFrame(medikamente_zeilen)

print('Berichte:', df_berichte.shape)             # (Zeilen, Spalten)
print('Nebenwirkungen:', df_nebenwirkungen.shape)
print('Medikamente:', df_medikamente.shape)

In [ ]:
# ---------- Speichern ----------
os.makedirs('data/processed', exist_ok=True)   # Ordner anlegen, falls es ihn noch nicht gibt
df_berichte.to_csv('data/processed/berichte.csv', index=False)
df_nebenwirkungen.to_csv('data/processed/nebenwirkungen.csv', index=False)
df_medikamente.to_csv('data/processed/medikamente.csv', index=False)

In [ ]:
# Berichte können doppelt vorkommen, wenn Tirzepatid und Semagltuig gegeben wurde... wie oft kommt das vor?
df_berichte['id'].duplicated().sum()   # wie oft kommt eine ID mehr als einmal vor?

In [ ]:
# ---------- Duplikate entfernen ----------
mask_doppelt = df_berichte['id'].duplicated()        # Maske: True, wenn die ID schon weiter oben vorkam
doppelte_ids = df_berichte[mask_doppelt]['id']       # aus diesen Zeilen nur die Spalte id nehmen

mask_berichte = df_berichte['id'].isin(doppelte_ids) & (df_berichte['abfrage'] == 'tirzepatide')   # doppelte Zeilen aus dem Tirzepatid-Download
df_berichte = df_berichte.drop(df_berichte[mask_berichte].index)   # diese Zeilen entfernen

mask_nebenwirkungen = df_nebenwirkungen['id'].isin(doppelte_ids) & (df_nebenwirkungen['abfrage'] == 'tirzepatide')
df_nebenwirkungen = df_nebenwirkungen.drop(df_nebenwirkungen[mask_nebenwirkungen].index)

mask_medikamente = df_medikamente['id'].isin(doppelte_ids) & (df_medikamente['abfrage'] == 'tirzepatide')
df_medikamente = df_medikamente.drop(df_medikamente[mask_medikamente].index)

# ---------- Kontrolle ----------
print('Berichte:', df_berichte.shape)             # (Zeilen, Spalten)
print('Nebenwirkungen:', df_nebenwirkungen.shape)
print('Medikamente:', df_medikamente.shape)
print('Doppelte IDs:', df_berichte['id'].duplicated().sum())   # sollte 0 sein

In [ ]:
# ---------- Spalte abfrage entfernen ----------
df_berichte = df_berichte.drop(columns='abfrage')             # Hilfsspalte entfernen
df_nebenwirkungen = df_nebenwirkungen.drop(columns='abfrage')
df_medikamente = df_medikamente.drop(columns='abfrage')

# ---------- Kontrolle ----------
print('Berichte:', df_berichte.shape)             
print('Nebenwirkungen:', df_nebenwirkungen.shape) 
print('Medikamente:', df_medikamente.shape)       

# ---------- Speichern ----------
df_berichte.to_csv('data/processed/berichte.csv', index=False)
df_nebenwirkungen.to_csv('data/processed/nebenwirkungen.csv', index=False)
df_medikamente.to_csv('data/processed/medikamente.csv', index=False)